In [2]:
# Importing necessary libraries
import numpy as np
import uuid
import umap
import shap
import pandas as pd
import faker
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import classification_report, accuracy_score
import datetime

from arize.pandas.logger import Client
from arize.utils.types import ModelTypes, Environments, Schema, Metrics, Embedding, EmbeddingColumnNames
import numpy as np
import pandas as pd
from faker import Faker


RANDOM_STATE = 123
import warnings

# Ignore all warnings
warnings.filterwarnings("ignore")

/Users/carterdaly/Documents/GitHub/data5580_hw/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def get_uuid():
    return uuid.uuid4().hex 
    
def random_dates(start, end, n=10):

    start_u = start.value//10**9
    end_u = end.value//10**9
    
    return pd.to_datetime(np.random.randint(start_u, end_u, n), unit='s')


def random_age(min_age=0, max_age=100):
    """
    Generate a random age within the given range.
    Args:
        min_age (int): Minimum possible age.
        max_age (int): Maximum possible age.
    Returns:
        int: Randomly generated age.
    """
    if not (isinstance(min_age, int) and isinstance(max_age, int)):
        raise ValueError("min_age and max_age must be integers.")
    if min_age < 0 or max_age < 0:
        raise ValueError("Ages cannot be negative.")
    if min_age > max_age:
        raise ValueError("min_age cannot be greater than max_age.")

    return random.randint(min_age, max_age)

In [4]:
fake = Faker()

In [5]:
# Load the breast cancer dataset
data = load_breast_cancer()

# Converting the dataset into a pandas DataFrame for easier handling
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target  # Add the target column (0 for malignant, 1 for benign)
df['target'] = df['target'].map({0: 'benign', 1: 'malignant'}) 

In [6]:
# Create a primary key for the dataset
df['prediction_id'] = [get_uuid() for _ in range(df.shape[0])]

start = pd.to_datetime('2026-01-01')
end = pd.to_datetime('2026-02-28')

df['event_timestamp'] = random_dates(start, end, df.shape[0])

df['state'] = [fake.state() for _ in range(df.shape[0])]

In [9]:
import random

df['age'] = [random_age() for _ in range(df.shape[0])]
df['zip_code'] = [fake.zipcode() for _ in range(df.shape[0])]

In [10]:
# Splitting the dataset into training and testing sets (80% training, 20% testing)
X = df.drop('target', axis=1)  # Features
y = df['target']               # Target

features = X.drop(['prediction_id', 'event_timestamp', 'state', 'age', 'zip_code'], axis=1).columns

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

In [11]:
# Initialize the Random Forest Classifier
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)

# Fit the model to the training data
rf_classifier.fit(X_train[features], y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [12]:
# Make predictions on the test set
y_pred = rf_classifier.predict(X_test[features])

# Evaluate the model's performance using accuracy and classification report
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

# Detailed classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.9912

Classification Report:
              precision    recall  f1-score   support

      benign       1.00      0.98      0.99        41
   malignant       0.99      1.00      0.99        73

    accuracy                           0.99       114
   macro avg       0.99      0.99      0.99       114
weighted avg       0.99      0.99      0.99       114



In [13]:
# Create the SHAP explainer for the trained model
explainer = shap.TreeExplainer(rf_classifier)

# Calculate SHAP values for the test set
shap_values = explainer.shap_values(X_test[features])

shap_values_positive_class = shap_values[:,:,1]

# 6. Convert the SHAP values to a DataFrame
shap_df = pd.DataFrame(shap_values_positive_class, columns=features)

# Because we are doing a bulk upload we need to rename the columns by added _shap at the end
shap_dataframe = pd.DataFrame(
        shap_values_positive_class, columns=[f"{fn}_shap" for fn in features], index=X_test.index
)

# Create a mapping
mapping = {fn: f"{fn}_shap" for fn in X[features].columns}

shap_dataframe.head()

,mean radius_shap,mean texture_shap,mean perimeter_shap,mean area_shap,mean smoothness_shap,mean compactness_shap,mean concavity_shap,mean concave points_shap,mean symmetry_shap,mean fractal dimension_shap,...,worst radius_shap,worst texture_shap,worst perimeter_shap,worst area_shap,worst smoothness_shap,worst compactness_shap,worst concavity_shap,worst concave points_shap,worst symmetry_shap,worst fractal dimension_shap
333,0.006227,0.012070,0.010931,0.011793,0.002703,0.002851,0.023857,0.051082,0.000684,0.000399,...,0.046181,0.008758,0.039769,0.054636,0.005563,0.007451,0.019355,0.041595,0.001851,0.000665
273,0.008022,0.007974,0.013161,0.017900,0.001117,0.002766,0.023050,0.047972,-0.000969,0.000877,...,0.047510,0.013139,0.043722,0.056637,-0.002086,0.006957,0.020438,0.042495,0.001245,0.001043
201,-0.023076,-0.002633,-0.033997,-0.033522,0.001392,-0.001851,-0.030186,-0.087250,-0.000824,-0.002244,...,-0.095546,-0.000588,-0.065963,-0.108055,0.000095,-0.003081,-0.020623,-0.088405,-0.000041,-0.000516
178,0.009886,-0.020028,0.009680,0.011953,0.009469,0.003560,0.024270,0.060509,0.001172,-0.007354,...,0.044889,0.000599,0.034879,0.049116,0.014614,0.007606,0.018536,0.041735,0.006710,-0.001883
85,-0.023417,-0.000158,-0.031711,-0.032024,-0.000701,0.002015,-0.035971,-0.085456,-0.001582,-0.000027,...,-0.094815,-0.006637,-0.067245,-0.110220,-0.001136,-0.001609,-0.015189,-0.073418,-0.002053,0.000660


In [14]:
embedder = umap.UMAP(random_state=RANDOM_STATE, n_components=3)

X_train_embeddings = embedder.fit_transform(X_train[features], y_train.map({'benign': 0, 'malignant': 1}))

In [15]:
mapping

{'mean radius': 'mean radius_shap',
 'mean texture': 'mean texture_shap',
 'mean perimeter': 'mean perimeter_shap',
 'mean area': 'mean area_shap',
 'mean smoothness': 'mean smoothness_shap',
 'mean compactness': 'mean compactness_shap',
 'mean concavity': 'mean concavity_shap',
 'mean concave points': 'mean concave points_shap',
 'mean symmetry': 'mean symmetry_shap',
 'mean fractal dimension': 'mean fractal dimension_shap',
 'radius error': 'radius error_shap',
 'texture error': 'texture error_shap',
 'perimeter error': 'perimeter error_shap',
 'area error': 'area error_shap',
 'smoothness error': 'smoothness error_shap',
 'compactness error': 'compactness error_shap',
 'concavity error': 'concavity error_shap',
 'concave points error': 'concave points error_shap',
 'symmetry error': 'symmetry error_shap',
 'fractal dimension error': 'fractal dimension error_shap',
 'worst radius': 'worst radius_shap',
 'worst texture': 'worst texture_shap',
 'worst perimeter': 'worst perimeter_shap'

In [16]:
tag_columns = ['state', 'age', 'zip_code']

In [17]:
embedding_feature_column_names = {
    # Dictionary keys will be the name of the embedding feature in the app
    "embedding": EmbeddingColumnNames(
        vector_column_name='embedding', # column containing embedding vector (required)
        # data_column_name="text", # column containing raw text (optional NLP)
        # link_to_data_column_name="image_link" # column containing image URL links (optional CV)
    )
}


# define schema 
schema = Schema(
    prediction_id_column_name="prediction_id",
    timestamp_column_name="event_timestamp",
    tag_column_names=tag_columns,
    shap_values_column_names=mapping,
    prediction_label_column_name="predicted_label",
    actual_label_column_name="actual_label",
    prediction_score_column_name="prediction_score",
    embedding_feature_column_names=embedding_feature_column_names,
    feature_column_names=features,
)


In [18]:
def create_upload_dataset(X, y_actual, y_pred, y_proba, features, model, explainer, embedder):
    df_test = X.copy()

    # Calculate SHAP values for the test set
    shap_values = explainer.shap_values(X[features])
    
    shap_values_positive_class = shap_values[:,:,1]
    
    # 6. Convert the SHAP values to a DataFrame
    shap_df = pd.DataFrame(shap_values_positive_class, columns=features)
    
    # Because we are doing a bulk upload we need to rename the columns by added _shap at the end
    shap_dataframe = pd.DataFrame(
            shap_values_positive_class, columns=[f"{fn}_shap" for fn in features], index=X.index
    )

    df_test = pd.concat([df_test, shap_dataframe], axis=1)
    df_test['actual_label'] = y_actual
    df_test['predicted_label'] = y_pred
    df_test['prediction_score'] = model.predict_proba(df_test[features])[:,1]
    df_test[['x','y','z']] = embedder.transform(df_test[features])
    df_test['embedding'] = df_test[['x','y','z']].values.tolist()
    
    df_test.reset_index(drop=True, inplace=True)

    return df_test


In [19]:
df_test = create_upload_dataset(X_test, y_test, rf_classifier.predict(X_test[features]), rf_classifier.predict_proba(X_test[features])[:,1],features, rf_classifier, explainer, embedder)

In [20]:
import os
from dotenv import load_dotenv

load_dotenv('/Users/carterdaly/Documents/GitHub/data5580_hw/.env')

True

In [21]:
schema

Schema(prediction_id_column_name='prediction_id', feature_column_names=Index(['mean radius', 'mean texture', 'mean perimeter', 'mean area',
       'mean smoothness', 'mean compactness', 'mean concavity',
       'mean concave points', 'mean symmetry', 'mean fractal dimension',
       'radius error', 'texture error', 'perimeter error', 'area error',
       'smoothness error', 'compactness error', 'concavity error',
       'concave points error', 'symmetry error', 'fractal dimension error',
       'worst radius', 'worst texture', 'worst perimeter', 'worst area',
       'worst smoothness', 'worst compactness', 'worst concavity',
       'worst concave points', 'worst symmetry', 'worst fractal dimension'],
      dtype='object'), tag_column_names=['state', 'age', 'zip_code'], timestamp_column_name='event_timestamp', prediction_label_column_name='predicted_label', prediction_score_column_name='prediction_score', actual_label_column_name='actual_label', actual_score_column_name=None, shap_value

In [22]:
df_test.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst concave points_shap,worst symmetry_shap,worst fractal dimension_shap,actual_label,predicted_label,prediction_score,x,y,z,embedding
0,11.250,14.78,71.38,390.0,0.08306,0.04458,0.000974,0.002941,0.1773,0.06081,...,0.041595,0.001851,0.000665,malignant,malignant,1.00,3.044728,-2.288504,6.335681,"[3.0447282791137695, -2.2885043621063232, 6.33..."
1,9.742,15.67,61.50,289.9,0.09037,0.04689,0.011030,0.014070,0.2081,0.06312,...,0.042495,0.001245,0.001043,malignant,malignant,1.00,3.429775,-6.437427,5.475908,"[3.429774761199951, -6.437427043914795, 5.4759..."
2,17.540,19.32,115.10,951.6,0.08968,0.11980,0.103600,0.074880,0.1506,0.05491,...,-0.088405,-0.000041,-0.000516,benign,benign,0.00,19.718023,-5.540632,23.879053,"[19.7180233001709, -5.5406317710876465, 23.879..."
3,13.010,22.22,82.01,526.4,0.06251,0.01938,0.001595,0.001852,0.1395,0.05234,...,0.041735,0.006710,-0.001883,malignant,malignant,0.97,1.095181,1.582217,5.226503,"[1.095180630683899, 1.5822173357009888, 5.2265..."
4,18.460,18.52,121.10,1075.0,0.09874,0.10530,0.133500,0.087950,0.2132,0.06022,...,-0.073418,-0.002053,0.000660,benign,benign,0.00,17.953123,-6.102934,24.163851,"[17.953123092651367, -6.102933883666992, 24.16..."


In [23]:
# create Arize client
SPACE_ID = "U3BhY2U6MzkyOTY6WDEyVw=="
API_KEY = os.getenv("ARIZE_API_KEY")
from arize.pandas.logger import Client
arize_client = Client(space_id=SPACE_ID, api_key=API_KEY)

batch_id = f'VALIDATION-{get_uuid()}'
print(f'Uploading {batch_id}')

#log data
response = arize_client.log(
    dataframe=df_test,
    schema=schema,
    model_id="binary-classification_4",
    model_version="1.0.0",
    model_type=ModelTypes.SCORE_CATEGORICAL,
    metrics_validation=[Metrics.CLASSIFICATION],
    validate=True,
    batch_id=f'{batch_id}',
    environment=Environments.VALIDATION
)

Uploading VALIDATION-ea577f233a2c433da0d0bce7614e02db
  arize.utils.logging | INFO | Success! Check out your data at https://app.arize.com/organizations/QWNjb3VudE9yZ2FuaXphdGlvbjozNzMyODpnWGlv/spaces/U3BhY2U6MzkyOTY6WDEyVw==/models/modelName/binary-classification_4?selectedTab=performance


In [24]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train[features], y_train)
# prediction = knn.predict(pd.DataFrame([feature_values]))

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None


In [25]:
# Make predictions on the test set
y_pred = knn.predict(X_test[features])

# Evaluate the model's performance using accuracy and classification report
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

# Detailed classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.9649

Classification Report:
              precision    recall  f1-score   support

      benign       0.97      0.93      0.95        41
   malignant       0.96      0.99      0.97        73

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114



In [26]:
from arize.api import Client

arize_client = Client(space_id=SPACE_ID, api_key=API_KEY) 

In [27]:
import random

record = {
    'prediction_id': get_uuid()
    , 'state': fake.state()
    , 'event_timestamp': random_dates(start, end, 1).to_pydatetime()[0]
}

for column in X[features]:
    record.update({column: float(random.uniform(min(X[column].values), max(X[column].values)))})
    # break

In [28]:
feature_values = {key: value for key, value in record.items() if key in features}

id = record['prediction_id']
print(f'logging {id}')

shap_values = explainer.shap_values(pd.DataFrame([feature_values]))
embedding = embedder.transform(pd.DataFrame([feature_values]))
embedding_payload = {
    "embedding": Embedding(
        vector = embedding.tolist()[0]
        # data: Union[str, List[str], NoneType] = None,
        # link_to_data: Optional[str] = None,
    )
}
label = rf_classifier.predict(pd.DataFrame([feature_values]))[0]
proba = rf_classifier.predict_proba(pd.DataFrame([feature_values]))[:,1][0]

response = arize_client.log(
        prediction_id=id,
        model_id="binary-classification_4",
        model_type=ModelTypes.SCORE_CATEGORICAL,
        environment=Environments.PRODUCTION,
        shap_values=dict(zip(feature_values.keys(), shap_values[:,:,1][0])),
        model_version='1.0.0',
        prediction_timestamp=int(record['event_timestamp'].timestamp()),
        features=feature_values,
        embedding_features=embedding_payload,
        prediction_label=(label, proba),
        tags={
            'age': random_age()
            , 'state': fake.state()
            , 'zip_code': fake.zipcode()
        }
)

res = response.result()
if res.status_code == 200:
    print('Success sending Prediction!')
else:
    print(f'Log failed with response code {res.status_code}, {res.text}')

logging 3305d5f0f85d42a087bc7c2ac077e20c
Success sending Prediction!


In [170]:
# list(set(df['target'].values.tolist()))

In [171]:
# sum(df['target'] == population[0]) / df.shape[0]

In [29]:
# log the actual
actual_response = arize_client.log(
    prediction_id=record['prediction_id'],
    model_id="binary-classification_4",
    model_type=ModelTypes.SCORE_CATEGORICAL,
    environment=Environments.PRODUCTION,
    actual_label=knn.predict(pd.DataFrame([feature_values]))[0],
)

# Listen to response code to ensure successful delivery
res = actual_response.result()
if res.status_code == 200:
    print('Success sending Actual!')
else:
    print(f'Log failed with response code {res.status_code}, {res.text}')

Success sending Actual!


In [30]:
start = pd.to_datetime('2026-03-01')
end = pd.to_datetime('2026-03-04')

In [31]:
for _ in range(500):
    record = {
        'prediction_id': get_uuid()
        , 'state': fake.state()
        , 'event_timestamp': random_dates(start, end, 1).to_pydatetime()[0]
    }
    
    for column in X[features]:
        record.update({column: float(random.uniform(min(X[column].values), max(X[column].values)))})

    feature_values = {key: value for key, value in record.items() if key in features}

    id = record['prediction_id']
    print(f'logging {id}')
    
    shap_values = explainer.shap_values(pd.DataFrame([feature_values]))
    embedding = embedder.transform(pd.DataFrame([feature_values]))
    embedding_payload = {
        "embedding": Embedding(
            vector = embedding.tolist()[0]
            # data: Union[str, List[str], NoneType] = None,
            # link_to_data: Optional[str] = None,
        )
    }
    label = rf_classifier.predict(pd.DataFrame([feature_values]))[0]
    proba = rf_classifier.predict_proba(pd.DataFrame([feature_values]))[:,1][0]
    
    response = arize_client.log(
            prediction_id=id,
            model_id="binary-classification_4",
            model_type=ModelTypes.SCORE_CATEGORICAL,
            environment=Environments.PRODUCTION,
            shap_values=dict(zip(feature_values.keys(), shap_values[:,:,1][0])),
            model_version='1.0.0',
            prediction_timestamp=int(record['event_timestamp'].timestamp()),
            features=feature_values,
            embedding_features=embedding_payload,
            prediction_label=(label, proba),
            tags={
                'age': random_age()
                , 'state': fake.state()
                , 'zip_code': fake.zipcode()
            }
    )
    
    res = response.result()
    if res.status_code == 200:
        print(f'Success sending Prediction for {id}')
    else:
        print(f'Log failed with response code {res.status_code}, {res.text}')

    actual_response = arize_client.log(
        prediction_id=record['prediction_id'],
        model_id="binary-classification_4",
        model_type=ModelTypes.SCORE_CATEGORICAL,
        environment=Environments.PRODUCTION,
        actual_label=knn.predict(pd.DataFrame([feature_values]))[0],
    )
    
    # Listen to response code to ensure successful delivery
    res = actual_response.result()
    if res.status_code == 200:
        print(f'Success sending Actual for {id}')
    else:
        print(f'Log failed with response code {res.status_code}, {res.text}')

logging 64cf707ac48447c6b164948d098660d3
Success sending Prediction for 64cf707ac48447c6b164948d098660d3
Success sending Actual for 64cf707ac48447c6b164948d098660d3
logging f62f4be38e094e5d902b242c10d7b0d4
Success sending Prediction for f62f4be38e094e5d902b242c10d7b0d4
Success sending Actual for f62f4be38e094e5d902b242c10d7b0d4
logging 46d3e7fc0987488eba712b918cd4b9f9
Success sending Prediction for 46d3e7fc0987488eba712b918cd4b9f9
Success sending Actual for 46d3e7fc0987488eba712b918cd4b9f9
logging 6336f39a5cb3475ab554effeafa65c3c
Success sending Prediction for 6336f39a5cb3475ab554effeafa65c3c
Success sending Actual for 6336f39a5cb3475ab554effeafa65c3c
logging 7792b2fdcb544056945d9103e6bd5c3a
Success sending Prediction for 7792b2fdcb544056945d9103e6bd5c3a
Success sending Actual for 7792b2fdcb544056945d9103e6bd5c3a
logging 7fff2c1e0d424b63b6adbd3ec9a912ee
Success sending Prediction for 7fff2c1e0d424b63b6adbd3ec9a912ee
Success sending Actual for 7fff2c1e0d424b63b6adbd3ec9a912ee
logging 38